# Pull Request Data Cleaning

---

In [2]:
# Import required libraries

import pandas as pd
import numpy as np
import os

In [3]:
# Check available files in the raw data folder

raw_path = "../data/raw"

print("Files available in raw data folder:")
print(os.listdir(raw_path))

Files available in raw data folder:
['all_pull_request.parquet', 'all_repository.parquet', 'all_user.parquet']


In [4]:
# Load the raw Pull Request dataset

df = pd.read_parquet("../data/raw/all_pull_request.parquet")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (2743854, 14)


In [5]:
# Inspect dataset structure

print("Dataset shape:")
print(df.shape)

print("\nDataset information:")
df.info()

print("\nColumn names:")
print(df.columns.tolist())

Dataset shape:
(2743854, 14)

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 2743854 entries, 0 to 2743853
Data columns (total 14 columns):
 #   Column      Dtype  
---  ------      -----  
 0   id          int64  
 1   number      int64  
 2   title       str    
 3   body        str    
 4   agent       str    
 5   user_id     int64  
 6   user        str    
 7   state       str    
 8   created_at  str    
 9   closed_at   str    
 10  merged_at   str    
 11  repo_id     float64
 12  repo_url    str    
 13  html_url    str    
dtypes: float64(1), int64(3), str(10)
memory usage: 3.3 GB

Column names:
['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url', 'html_url']


In [6]:
# Check missing values before handling them

missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

print("\nMissing value percentages:")
print((df.isnull().sum() / len(df) * 100).round(2))

Missing values by column:
id                 0
number             0
title              1
body           21481
agent              0
user_id            0
user               0
state              0
created_at         0
closed_at     292402
merged_at     533798
repo_id         8902
repo_url           0
html_url           0
dtype: int64

Missing value percentages:
id             0.00
number         0.00
title          0.00
body           0.78
agent          0.00
user_id        0.00
user           0.00
state          0.00
created_at     0.00
closed_at     10.66
merged_at     19.45
repo_id        0.32
repo_url       0.00
html_url       0.00
dtype: float64


In [7]:
# Remove the single Pull Request with a missing title

before = len(df)

df = df.dropna(subset=["title"]).copy()

after = len(df)

print("Rows removed:", before - after)
print("Rows remaining:", after)
print("Missing title values:", df["title"].isna().sum())

Rows removed: 1
Rows remaining: 2743853
Missing title values: 0


In [8]:
# Replace missing Pull Request bodies with empty strings

df["body"] = df["body"].fillna("")

print("Missing body values:", df["body"].isna().sum())

Missing body values: 0


In [9]:
# Keep rows with missing repo_id
# Repository information is still available through repo_url

print("Missing repo_id values retained:", df["repo_id"].isna().sum())
print("Missing repo_url values:", df["repo_url"].isna().sum())

Missing repo_id values retained: 8902
Missing repo_url values: 0


In [10]:
# Check missing closed_at and merged_at by Pull Request state

print("Missing closed_at by state:")
print(df.groupby("state")["closed_at"].apply(lambda x: x.isna().sum()))

print("\nMissing merged_at by state:")
print(df.groupby("state")["merged_at"].apply(lambda x: x.isna().sum()))

Missing closed_at by state:
state
closed         0
open      292402
Name: closed_at, dtype: int64

Missing merged_at by state:
state
closed    241395
open      292402
Name: merged_at, dtype: int64


In [11]:
# Check for duplicate Pull Request IDs after the cleaning steps.

duplicate_pr_ids = df["id"].duplicated().sum()

print("Duplicate PR IDs:", duplicate_pr_ids)

Duplicate PR IDs: 0


In [12]:
# Convert Pull Request timestamp columns to datetime

date_columns = ["created_at", "closed_at", "merged_at"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

print("Data types after conversion:")
print(df[date_columns].dtypes)

Data types after conversion:
created_at    datetime64[us, UTC]
closed_at     datetime64[us, UTC]
merged_at     datetime64[us, UTC]
dtype: object


In [13]:
# Check for date-order issues

date_order_issues = df[
    (
        df["closed_at"].notna() &
        (df["closed_at"] < df["created_at"])
    )
    |
    (
        df["merged_at"].notna() &
        (df["merged_at"] < df["created_at"])
    )
    |
    (
        df["merged_at"].notna() &
        df["closed_at"].notna() &
        (df["merged_at"] > df["closed_at"])
    )
]

print("Date-order issues:", len(date_order_issues))

print("\nSample date-order issues:")
print(
    date_order_issues[
        ["id", "state", "created_at", "closed_at", "merged_at"]
    ].head(10)
)

Date-order issues: 50

Sample date-order issues:
                id   state                created_at  \
85059   3338498648  closed 2025-08-20 15:01:22+00:00   
90656   3382047981  closed 2025-09-04 04:03:01+00:00   
105043  3521453239  closed 2025-10-16 11:23:55+00:00   
110165  3540468626  closed 2025-10-22 11:27:14+00:00   
145520  3519282037  closed 2025-10-15 19:21:21+00:00   
146412  3472006404  closed 2025-10-01 04:51:18+00:00   
164767  3176607539  closed 2025-06-25 18:40:02+00:00   
207457  3496753851  closed 2025-10-08 20:14:11+00:00   
281418  3293951306  closed 2025-08-05 17:51:42+00:00   
327539  3443629185  closed 2025-09-23 04:27:17+00:00   

                       closed_at                 merged_at  
85059  2025-08-20 15:05:00+00:00 2025-08-20 15:10:28+00:00  
90656  2025-09-04 06:29:52+00:00 2025-09-04 06:29:55+00:00  
105043 2025-10-16 11:42:57+00:00 2025-10-16 11:42:58+00:00  
110165 2025-10-22 11:45:02+00:00 2025-10-22 11:45:04+00:00  
145520 2025-10-15 20:19:19+00

In [14]:
# Check for empty or whitespace-only text values

empty_titles = df["title"].fillna("").astype(str).str.strip().eq("").sum()
empty_bodies = df["body"].fillna("").astype(str).str.strip().eq("").sum()

print("Empty or whitespace-only titles:", empty_titles)
print("Empty or whitespace-only bodies:", empty_bodies)

Empty or whitespace-only titles: 0
Empty or whitespace-only bodies: 21481


In [15]:
# Check unique values in categorical columns

print("AI Coding Agents:")
print(df["agent"].value_counts(dropna=False))

print("\nPull Request States:")
print(df["state"].value_counts(dropna=False))

AI Coding Agents:
agent
OpenAI_Codex    2069595
Copilot          349694
Cursor           212544
Google_Jules      50490
Devin             43298
Claude_Code       18232
Name: count, dtype: int64

Pull Request States:
state
closed    2451451
open       292402
Name: count, dtype: int64


In [16]:
# Final data cleaning validation

print("Final dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate PR IDs:", df["id"].duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nPR states:")
print(df["state"].value_counts())

print("\nAI coding agents:")
print(df["agent"].value_counts())

Final dataset shape: (2743853, 14)

Missing values:
id                 0
number             0
title              0
body               0
agent              0
user_id            0
user               0
state              0
created_at         0
closed_at     292402
merged_at     533797
repo_id         8902
repo_url           0
html_url           0
dtype: int64

Duplicate PR IDs: 0

Data types:
id                          int64
number                      int64
title                         str
body                          str
agent                         str
user_id                     int64
user                          str
state                         str
created_at    datetime64[us, UTC]
closed_at     datetime64[us, UTC]
merged_at     datetime64[us, UTC]
repo_id                   float64
repo_url                      str
html_url                      str
dtype: object

PR states:
state
closed    2451451
open       292402
Name: count, dtype: int64

AI coding agents:
agent
OpenAI_Codex

In [17]:
# Save the cleaned dataset to the processed data directory

output_file = "../data/processed/cleaned_pull_requests.parquet"

df.to_parquet(output_file, index=False)

print("Cleaned dataset saved successfully.")
print("File:", output_file)
print("Shape:", df.shape)

Cleaned dataset saved successfully.
File: ../data/processed/cleaned_pull_requests.parquet
Shape: (2743853, 14)


---

# Repository Data Cleaning

In [18]:
# Load and inspect the raw repository dataset before cleaning.

repository_df = pd.read_parquet("../data/raw/all_repository.parquet")

print("Repository dataset shape:", repository_df.shape)

print("\nMissing values:")
print(repository_df.isnull().sum())

Repository dataset shape: (326798, 8)

Missing values:
id                0
url               0
license      221769
full_name         0
is_forked         0
language      44681
forks             0
stars             0
dtype: int64


In [19]:
# Check repository IDs, numeric values, and basic validity before cleaning.

print("Duplicate repository IDs:", repository_df["id"].duplicated().sum())

print("Negative forks:", (repository_df["forks"] < 0).sum())
print("Negative stars:", (repository_df["stars"] < 0).sum())

print("Missing repository URLs:", repository_df["url"].isna().sum())
print("Missing repository names:", repository_df["full_name"].isna().sum())

Duplicate repository IDs: 0
Negative forks: 0
Negative stars: 0
Missing repository URLs: 0
Missing repository names: 0


## Handling Missing Repository Metadata

The `license` and `language` columns contain missing values because this information is not available for some repositories.

These missing values were not replaced with artificial values because filling them with assumptions could introduce incorrect information into the dataset.

The repository records are therefore retained with these values as missing. During Feature Engineering, appropriate techniques such as categorical encoding and missing-value handling can be applied if these columns are used as machine learning features.

The original repository information is preserved without unnecessary modification.

In [20]:
# Confirm repository metadata missing values are retained and repository records are unchanged.

print("Missing license values:", repository_df["license"].isna().sum())
print("Missing language values:", repository_df["language"].isna().sum())
print("Repository rows retained:", len(repository_df))

Missing license values: 221769
Missing language values: 44681
Repository rows retained: 326798


In [21]:
# Save the cleaned repository dataset to the processed data directory.

repository_output = "../data/processed/cleaned_repositories.parquet"

repository_df.to_parquet(repository_output, index=False)

print("Cleaned repository dataset saved successfully.")
print("File:", repository_output)
print("Shape:", repository_df.shape)

Cleaned repository dataset saved successfully.
File: ../data/processed/cleaned_repositories.parquet
Shape: (326798, 8)


---

# User Data Cleaning

In [22]:
# Load and inspect the raw user dataset before cleaning.

user_df = pd.read_parquet("../data/raw/all_user.parquet")

print("User dataset shape:", user_df.shape)

print("\nMissing values:")
print(user_df.isnull().sum())

User dataset shape: (161255, 5)

Missing values:
id            50
login          0
followers     50
following     50
created_at    50
dtype: int64


In [23]:
# Check user IDs, numeric values, and basic validity before cleaning.

print("Missing user IDs:", user_df["id"].isna().sum())
print("Duplicate valid user IDs:", user_df["id"].dropna().duplicated().sum())

print("Negative followers:", (user_df["followers"] < 0).sum())
print("Negative following:", (user_df["following"] < 0).sum())

print("Missing login values:", user_df["login"].isna().sum())

Missing user IDs: 50
Duplicate valid user IDs: 0
Negative followers: 0
Negative following: 0
Missing login values: 0


In [24]:
# Check user account creation dates for missing, invalid, or future values.

user_created_at = pd.to_datetime(
    user_df["created_at"],
    errors="coerce",
    utc=True
)

print("Missing/invalid created_at:", user_created_at.isna().sum())
print("Future created_at:", (user_created_at > pd.Timestamp.now(tz="UTC")).sum())

Missing/invalid created_at: 50
Future created_at: 0


## Handling Missing User Metadata

The user dataset contains 50 records with missing `id`, `followers`, `following`, and `created_at` values.

These records are retained because the missing values represent unavailable user information rather than invalid user records. No artificial values are inserted during data cleaning.

When user information is used for feature engineering, missing values will be handled appropriately based on the specific feature and machine learning requirement.

In [25]:
# Confirm that the user records with missing metadata are retained.

print("User records retained:", len(user_df))
print("Missing user IDs:", user_df["id"].isna().sum())
print("Missing followers:", user_df["followers"].isna().sum())
print("Missing following:", user_df["following"].isna().sum())
print("Missing created_at:", user_df["created_at"].isna().sum())

User records retained: 161255
Missing user IDs: 50
Missing followers: 50
Missing following: 50
Missing created_at: 50


In [26]:
# Save the cleaned user dataset to the processed data directory.

user_output = "../data/processed/cleaned_users.parquet"

user_df.to_parquet(user_output, index=False)

print("Cleaned user dataset saved successfully.")
print("File:", user_output)
print("Shape:", user_df.shape)

Cleaned user dataset saved successfully.
File: ../data/processed/cleaned_users.parquet
Shape: (161255, 5)


---